In [1]:
import os
from brian2 import *
sys.path.append('Neuron and Synapse Models')
from neuronModels import *
from ringAttractorClass import *

sys.path.append('Tools')
from plottingTools import *
from utils import *

set_device('cpp_standalone', build_on_run=False)
dtMujoco = 0.001  # time step for mujoco simulation

In [2]:
# Get the absolute path to the project root using the notebook's working directory
project_root = os.getcwd()

# Create absolute paths
build_dir = os.path.join(project_root, 'standalone_build_mujoco')
tools_dir = os.path.join(project_root, 'Tools')
cpp_path = os.path.join(tools_dir, 'socket_input.cpp')
header_path = os.path.join(tools_dir, 'socket_input.h')

# Print paths for verification
print(f"Project root: {project_root}")
print(f"Build directory: {build_dir}")
print(f"CPP file path: {cpp_path}")
print(f"Header file path: {header_path}")

# Verify file existence
if not os.path.exists(cpp_path):
    raise FileNotFoundError(f"Could not find {cpp_path}")
if not os.path.exists(header_path):
    raise FileNotFoundError(f"Could not find {header_path}")

Project root: /home/bmaacaron-iit.local/Documents/GitRepos/JointAttractorNets
Build directory: /home/bmaacaron-iit.local/Documents/GitRepos/JointAttractorNets/standalone_build_mujoco
CPP file path: /home/bmaacaron-iit.local/Documents/GitRepos/JointAttractorNets/Tools/socket_input.cpp
Header file path: /home/bmaacaron-iit.local/Documents/GitRepos/JointAttractorNets/Tools/socket_input.h


In [3]:
@implementation(
    'cpp',
    '// code lives in socket_input.cpp',
    sources=[cpp_path],
    headers=['"{}"'.format(header_path)],
    # Fix the HOST_IP string literal by escaping quotes
    define_macros=[('HOST_IP', '\\"0.0.0.0\\"'), ('PORT_NUM', 5005), ('BUF_BYTES', 1024)],
    include_dirs=[tools_dir])
@check_units(index=1, result=1)
def get_socket_sample(socket_index: int) -> float:
    """Brian only uses the C++ version in standalone mode."""
    raise NotImplementedError

In [4]:
eqs = '''
x0 = get_socket_sample(0) : 1
x1 = get_socket_sample(1) : 1
x2 = get_socket_sample(2) : 1
x3 = get_socket_sample(3) : 1
'''

neuronGroup = NeuronGroup(1, eqs, method='euler')
monitored_vars = ['x0', 'x1', 'x2', 'x3']
mon = StateMonitor(neuronGroup, monitored_vars, record=True)
run(20*second, report='text')

In [5]:
device.build(directory = build_dir, compile=True, run=False, debug=True, clean=True)

g++ -c -Wno-write-strings -DHOST_IP=\"0.0.0.0\" -DPORT_NUM=5005 -DBUF_BYTES=1024 -I"/home/bmaacaron-iit.local/.virtualenvs/JointAttractorNets-Brian2/include" -I"/home/bmaacaron-iit.local/Documents/GitRepos/JointAttractorNets/Tools" -w -O3 -ffast-math -fno-finite-math-only -march=native -std=c++11 -I.  -g -DDEBUG /home/bmaacaron-iit.local/Documents/GitRepos/JointAttractorNets/Tools/socket_input.cpp -o /home/bmaacaron-iit.local/Documents/GitRepos/JointAttractorNets/Tools/socket_input.o
g++ -c -Wno-write-strings -DHOST_IP=\"0.0.0.0\" -DPORT_NUM=5005 -DBUF_BYTES=1024 -I"/home/bmaacaron-iit.local/.virtualenvs/JointAttractorNets-Brian2/include" -I"/home/bmaacaron-iit.local/Documents/GitRepos/JointAttractorNets/Tools" -w -O3 -ffast-math -fno-finite-math-only -march=native -std=c++11 -I.  -g -DDEBUG code_objects/statemonitor_codeobject.cpp -o code_objects/statemonitor_codeobject.o
g++ -c -Wno-write-strings -DHOST_IP=\"0.0.0.0\" -DPORT_NUM=5005 -DBUF_BYTES=1024 -I"/home/bmaacaron-iit.local/.vir

In [6]:
for i in range(10):
    device.run()
    print(f"Time is: x0 - {mon.x0},\n Counter is: x1 - {mon.x1},\n Input Angle is: x2 - {mon.x2},\n Velocity is: x3 - {mon.x3}")

Setting results dir to '/home/bmaacaron-iit.local/Documents/GitRepos/JointAttractorNets/standalone_build_mujoco/results/'
Starting simulation at t=0 s for duration 20 s
Server listening on 0.0.0.0:5005
Connection established with 10.240.78.167:42278
Time: 0.495s, Counter: 0, Position: 30.7105°, Velocity: -19.7212°/s
Time: 0.545s, Counter: 1, Position: 31.57°, Velocity: -15.4091°/s
Time: 0.595s, Counter: 2, Position: 32.3721°, Velocity: -15.4922°/s
Time: 0.645s, Counter: 3, Position: 33.2316°, Velocity: -15.585°/s
Time: 0.695s, Counter: 4, Position: 34.091°, Velocity: -15.5266°/s
Time: 0.745s, Counter: 5, Position: 34.8931°, Velocity: -15.5037°/s
Time: 0.795s, Counter: 6, Position: 35.7526°, Velocity: -15.5019°/s
Time: 0.845s, Counter: 7, Position: 36.612°, Velocity: -15.5128°/s
Time: 0.895s, Counter: 8, Position: 37.4141°, Velocity: -15.5277°/s
Time: 0.945s, Counter: 9, Position: 38.2736°, Velocity: -15.5312°/s
Time: 0.995s, Counter: 10, Position: 39.133°, Velocity: -15.5312°/s
Time: 1

KeyboardInterrupt: 